<a href="https://colab.research.google.com/github/amann45/AI_Labworks/blob/main/Lab7/lab7RNN_Encoder_Decoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Additive Attention [Aman Kumar Ray: ACE080BCT010]

## Objective: To implement an additive attention-based RNN Encoder–Decoder model for machine translation.

## Theory

The Attention Mechanism is a technique used in Encoder–Decoder Recurrent Neural Networks (RNNs) to improve the performance of sequence-to-sequence (Seq2Seq) models. It was introduced to overcome the limitation of traditional encoder–decoder architectures, where the encoder compresses the entire input sequence into a single fixed-length context vector. This compression often causes information loss, especially for long input sequences.

Instead of relying on only one context vector, the attention mechanism allows the decoder to focus on the most relevant parts of the input sequence while generating each output token.

--------------------------------------------------------------------------------

## Why is Attention Needed?

In a basic Encoder–Decoder RNN:

1. The encoder reads the entire input sequence.
2. The final hidden state becomes the context vector.
3. The decoder uses only this context vector to generate the output.

### Problem

For long sentences:

• Important information may be forgotten.
• Earlier words have less influence on the final hidden state.
• Translation accuracy decreases.

Example:

Input (English):

"The little boy who was playing football in the park yesterday is my brother."

The encoder has to compress the entire sentence into one vector, making it difficult to preserve every important detail.

Attention solves this problem by allowing the decoder to access all encoder hidden states instead of only the final hidden state.


## Basic Encoder–Decoder without Attention

In a basic Encoder–Decoder RNN, the encoder reads the entire input sequence and compresses it into a single fixed-length context vector. The decoder uses only this context vector to generate the output sequence.

```text
                 Input Sequence

x1 ───► x2 ───► x3 ───► x4
              Encoder

h1 ───► h2 ───► h3 ───► h4
                             │
                             │
                     Context Vector
                             │
                             ▼
                         Decoder
                             │
                             ▼
                 y1 ───► y2 ───► y3
```

**Note:** Only the final hidden state (**h4**) is passed to the decoder.

---

## Encoder with Attention

Instead of storing only the final hidden state, the encoder stores **all hidden states**, allowing the decoder to access information from every input word.

```text
               Input Sequence

x1 ───► x2 ───► x3 ───► x4
              Encoder

h1 ───► h2 ───► h3 ───► h4

Stored Encoder Hidden States:

[h1, h2, h3, h4]
```

These hidden states are passed to the **Attention Mechanism**.

---

## Decoder with Attention

When generating each output word, the decoder performs the following steps:

1. Looks at all encoder hidden states.
2. Computes attention scores.
3. Calculates attention weights.
4. Creates a context vector.
5. Generates the next output word.

```text
           Encoder Hidden States

      h1       h2       h3       h4
       │        │        │        │
       └────────┼────────┼────────┘
                │
                ▼
         Attention Mechanism
                │
                ▼
       Context Vector (Ct)
                │
                ▼
        Decoder Hidden State
                │
                ▼
         Next Output Word
```

For **every output word**, the decoder computes a new context vector based on the attention weights. This enables the model to focus on the most relevant input words instead of relying only on the final encoder hidden state.

In [26]:
# requirements
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.set_default_device(device)
print(f"Using device = {torch.get_default_device()}")

Using device = cpu


In [27]:
SOS_token = 0 # Start of the Sentence
EOS_token = 1 # End of the Sentence

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

In [28]:
# Turn a Unicode string to plain ASCII, thanks to
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, trim, and remove non-letter characters
def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s)
    return s.strip()

In [29]:
def readLangs(path:str):
    lang1 = 'eng'; lang2 = 'fra'
    print("Reading lines...")

    # Read the file and split into lines
    lines = open(path, encoding='utf-8').\
        read().strip().split('\n')

    # Split every line into pairs and normalize (english to french)
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

    # Reverse pairs: English-French -> French-English
    pairs = [list(reversed(p)) for p in pairs]

    # Input is French, output is English
    input_lang = Lang(lang2)
    output_lang = Lang(lang1)

    return input_lang, output_lang, pairs

In [30]:
MAX_LENGTH = 5

eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)


def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

In [31]:
def prepareData(path):
    input_lang, output_lang, pairs = readLangs(path)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

In [32]:
PATH = r'/content/data.txt'

input_lang, output_lang, pairs = prepareData(PATH)
print(random.choice(pairs))

output_lang.word2index['am']  # try different English words.

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
['vous etes tres contraries', 'you re very upset']


15

In [33]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.rnn(embedded)
        return output, hidden

In [34]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super(BahdanauAttention, self).__init__()
        self.Wa = nn.Linear(hidden_size, hidden_size)
        self.Ua = nn.Linear(hidden_size, hidden_size)
        self.Va = nn.Linear(hidden_size, 1)

    def forward(self, query, keys):
        scores = self.Va(torch.tanh(self.Wa(query) + self.Ua(keys)))
        scores = scores.squeeze(2).unsqueeze(1)

        weights = F.softmax(scores, dim=-1)
        context = torch.bmm(weights, keys)

        return context, weights

In [35]:
class AttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.1):
        super(AttnDecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.attention = BahdanauAttention(hidden_size)
        self.rnn = nn.RNN(2 * hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []
        attentions = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden, attn_weights = self.forward_step(
                decoder_input, decoder_hidden, encoder_outputs
            )
            decoder_outputs.append(decoder_output)
            attentions.append(attn_weights)

            if target_tensor is not None:
                # Teacher forcing: Feed the target as the next input
                decoder_input = target_tensor[:, i].unsqueeze(1) # Teacher forcing
            else:
                # Without teacher forcing: use its own predictions as the next input
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()  # detach from history as input

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        attentions = torch.cat(attentions, dim=1)

        return decoder_outputs, decoder_hidden, attentions


    def forward_step(self, input, hidden, encoder_outputs):
        embedded =  self.dropout(self.embedding(input))

        query = hidden.permute(1, 0, 2) # seq_len, batch, hidden_size -> batch, seq_len, hidden_size
        context, attn_weights = self.attention(query, encoder_outputs)
        input_rnn = torch.cat((embedded, context), dim=2)

        output, hidden = self.rnn(input_rnn, hidden)
        output = self.out(output)

        return output, hidden, attn_weights

In [36]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepareData(path=PATH)

    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids

    train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))

    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
    return input_lang, output_lang, train_dataloader

### Training Loop

In [37]:
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
          decoder_optimizer, criterion):

    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor) # using teacher forcing

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [38]:
import time
import math

def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

In [39]:
import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker
import numpy as np

def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    # this locator puts ticks at regular intervals
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)

In [40]:
def train(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
               print_every=100, plot_every=100):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                        epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)

## Evaluation Code

In [41]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn

In [42]:
def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words,attentions = evaluate(encoder, decoder, pair[0], input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print("\nAttention Matrix:")
        print(attentions)

        print()

### Training and Evaluating

In [43]:
hidden_size = 128
batch_size = 32
EPOCHS = 200

input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = AttnDecoderRNN(hidden_size, output_lang.n_words).to(device)

train(train_dataloader, encoder, decoder, EPOCHS, print_every=5, plot_every=5)

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
0m 22s (- 14m 45s) (5 2%) 1.8140
0m 37s (- 12m 0s) (10 5%) 1.0599
0m 53s (- 10m 59s) (15 7%) 0.7370
1m 8s (- 10m 19s) (20 10%) 0.4990
1m 24s (- 9m 49s) (25 12%) 0.3297
1m 40s (- 9m 28s) (30 15%) 0.2143
1m 56s (- 9m 9s) (35 17%) 0.1455
2m 11s (- 8m 46s) (40 20%) 0.1060
2m 26s (- 8m 25s) (45 22%) 0.0844
2m 41s (- 8m 5s) (50 25%) 0.0716
2m 56s (- 7m 45s) (55 27%) 0.0630
3m 11s (- 7m 26s) (60 30%) 0.0591
3m 26s (- 7m 9s) (65 32%) 0.0579
3m 41s (- 6m 51s) (70 35%) 0.0535
3m 56s (- 6m 33s) (75 37%) 0.0516
4m 10s (- 6m 16s) (80 40%) 0.0504
4m 25s (- 5m 59s) (85 42%) 0.0493
4m 40s (- 5m 42s) (90 45%) 0.0464
4m 55s (- 5m 26s) (95 47%) 0.0462
5m 10s (- 5m 10s) (100 50%) 0.0480
5m 25s (- 4m 54s) (105 52%) 0.0462
5m 40s (- 4m 38s) (110 55%) 0.0461
5m 56s (- 4m 23s) (115 57%) 0.0440
6m 11s (- 4m 7s) (120 60%) 0.0428
6m 26s (- 3m 51s) (125 62%) 0.0441
6m 41s (- 3m 36s) (130 65

In [44]:
encoder.eval()
decoder.eval()
evaluateRandomly(encoder, decoder)

> je fais la diete
= i m fasting
< i m fasting <EOS>

Attention Matrix:
tensor([[[5.0967e-01, 1.0700e-01, 1.5683e-01, 2.0025e-01, 2.6245e-02],
         [2.1276e-04, 1.9845e-01, 4.3092e-01, 3.6980e-01, 6.1109e-04],
         [4.1287e-04, 1.1417e-01, 1.1489e-01, 7.6070e-01, 9.8254e-03],
         [1.6077e-02, 4.0096e-01, 1.9090e-01, 3.8616e-01, 5.8961e-03],
         [4.8960e-03, 2.6850e-02, 2.2329e-01, 2.5472e-02, 7.1949e-01]]])

> vous etes jeune
= you re young
< you re young <EOS>

Attention Matrix:
tensor([[[2.0022e-02, 8.2187e-01, 5.0874e-02, 1.0724e-01],
         [1.9878e-04, 3.1018e-03, 9.9258e-01, 4.1213e-03],
         [6.1524e-05, 7.6283e-04, 9.9382e-01, 5.3606e-03],
         [8.9267e-04, 1.5804e-02, 8.5838e-01, 1.2493e-01],
         [1.2208e-05, 7.2059e-04, 3.4742e-01, 6.5185e-01]]])

> nous sommes sournois
= we re sneaky
< we re sneaky <EOS>

Attention Matrix:
tensor([[[0.7946, 0.1566, 0.0311, 0.0176],
         [0.0344, 0.0270, 0.8509, 0.0877],
         [0.0051, 0.0056, 0.9471, 0

## Discussion

The attention-based RNN Encoder–Decoder model was successfully implemented and trained for French-to-English machine translation. The training loss decreased steadily throughout the 200 epochs, indicating effective learning and convergence of the model. During evaluation, the model correctly translated most of the test sentences, while a few predictions were semantically similar or slightly different from the reference translations. The generated attention matrices demonstrated that the decoder focused on the most relevant input words when producing each output word, confirming the effectiveness of the Bahdanau Attention mechanism. Overall, the results show that attention improves translation accuracy and provides better alignment between the source and target sentences.

## Conclusion

This lab successfully demonstrated the implementation of an attention-based RNN Encoder–Decoder model using the Bahdanau Attention mechanism. The model achieved accurate French-to-English translations with a low training loss and effectively learned meaningful word alignments through attention. The attention mechanism overcame the limitations of the traditional Encoder–Decoder architecture by allowing the decoder to access all encoder hidden states during translation. This lab highlights the importance of attention in improving s2s learning and serves as a foundation for modern neural machine translation and Transformer-based models.